# PPE Detection Workbench

Upload factory-floor images and run **YOLO PPE detection** (hardhat, safety-vest, goggles) via the in-cluster `yolo-ppe-serving` endpoint or compare with a local Ultralytics model.

**Env vars:** `PPE_ENDPOINT`, `NEUROFACE_URL`, `PPE_CONFIDENCE`

In [ ]:
import os
import json
from pathlib import Path

import cv2
import numpy as np
import requests
from IPython.display import display, Image as IPImage
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle

PPE_ENDPOINT = os.environ.get("PPE_ENDPOINT", "http://yolo-ppe-serving:8080").rstrip("/")
NEUROFACE_URL = os.environ.get("NEUROFACE_URL", "https://neuroface.apps.cluster.example.com").rstrip("/")
PPE_CONFIDENCE = float(os.environ.get("PPE_CONFIDENCE", "0.5"))
PPE_CLASSES = [c.strip() for c in os.environ.get("PPE_CLASSES", "hardhat,safety-vest,goggles").split(",") if c.strip()]

COLORS = {
    "hardhat": "#f4c430",
    "safety-vest": "#ff6b35",
    "goggles": "#4ecdc4",
    "person": "#95a5a6",
}

print("PPE endpoint:", PPE_ENDPOINT)
print("NeuroFace URL:", NEUROFACE_URL)
print("Classes:", PPE_CLASSES)
print("Confidence:", PPE_CONFIDENCE)

health = requests.get(f"{PPE_ENDPOINT}/health", timeout=30)
print("Serving health:", health.status_code, health.json())

status = requests.get(f"{NEUROFACE_URL}/api/ppe/status", timeout=30, verify=False)
print("NeuroFace PPE status:", status.status_code, status.text[:200])

## 1. Upload an image

Place a `.jpg` or `.png` in `uploads/` or set `IMAGE_PATH` below.

In [ ]:
UPLOAD_DIR = Path("uploads")
UPLOAD_DIR.mkdir(exist_ok=True)

# Change this path to your image
IMAGE_PATH = UPLOAD_DIR / "sample.jpg"

if IMAGE_PATH.exists():
    display(IPImage(filename=str(IMAGE_PATH), width=480))
else:
    print(f"Add an image at {IMAGE_PATH} (or update IMAGE_PATH) then re-run this cell.")

## 2. Detect PPE via yolo-ppe-serving

In [ ]:
def detect_ppe(image_path: Path, confidence: float = PPE_CONFIDENCE) -> list:
    with open(image_path, "rb") as f:
        payload = f.read()
    response = requests.post(
        f"{PPE_ENDPOINT}/v1/predict",
        data=payload,
        headers={"X-Confidence": str(confidence)},
        timeout=120,
    )
    response.raise_for_status()
    return response.json().get("detections", [])


if IMAGE_PATH.exists():
    detections = detect_ppe(IMAGE_PATH)
    print(json.dumps(detections, indent=2))
else:
    detections = []
    print("No image loaded.")

## 3. Visualize bounding boxes

In [ ]:
def draw_detections(image_path: Path, detections: list):
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(img_rgb)
    ax.axis("off")
    for det in detections:
        bbox = det.get("bbox", {})
        x1, y1, x2, y2 = bbox.get("x1"), bbox.get("y1"), bbox.get("x2"), bbox.get("y2")
        if None in (x1, y1, x2, y2):
            continue
        name = det.get("class_name", "unknown")
        conf = det.get("confidence", 0)
        color = COLORS.get(name, "#e74c3c")
        rect = Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, max(y1 - 5, 0), f"{name} {conf:.2f}", color=color, fontsize=10, weight="bold")
    plt.show()


if IMAGE_PATH.exists() and detections:
    draw_detections(IMAGE_PATH, detections)
elif IMAGE_PATH.exists():
    print("No detections above confidence threshold.")
else:
    print("Load an image first.")

## 4. Compare with local Ultralytics model (optional)

Downloads `best.pt` from HuggingFace on first run. CPU inference — may take a few minutes.

In [ ]:
LOCAL_MODEL = Path("models/best.pt")
LOCAL_MODEL.parent.mkdir(exist_ok=True)

if not LOCAL_MODEL.exists():
    from huggingface_hub import hf_hub_download
    import shutil

    src = hf_hub_download(repo_id="Hexmon/vyra-yolo-ppe-detection", filename="best.pt")
    shutil.copy(src, LOCAL_MODEL)
    print("Downloaded model to", LOCAL_MODEL)

if IMAGE_PATH.exists():
    from ultralytics import YOLO

    model = YOLO(str(LOCAL_MODEL))
    results = model.predict(str(IMAGE_PATH), conf=PPE_CONFIDENCE, verbose=False)
    local_detections = []
    for result in results:
        if result.boxes is None:
            continue
        for box in result.boxes:
            cls_id = int(box.cls[0])
            name = model.names.get(cls_id, str(cls_id))
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            local_detections.append({
                "class_name": name,
                "confidence": float(box.conf[0]),
                "bbox": {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
            })
    print("Local model detections:", len(local_detections))
    draw_detections(IMAGE_PATH, local_detections)
else:
    print("Load an image first.")